In [0]:
UC_CATALOG = "preethiworkspace_7405608993331015"
UC_SCHEMA  = "default"
VOLUME_NAME = "bank_fraud"

BASE = f"/Volumes/preethiworkspace_7405608993331015/default/bank_fraud"

paths = {
  "landing":      f"{BASE}/landing/transactions",
  "schemaLoc":    f"{BASE}/schema/transactions",
  "ckpt_bronze":  f"{BASE}/checkpoints/bronze",
  "ckpt_silver":  f"{BASE}/checkpoints/silver",
  "ckpt_gold":    f"{BASE}/checkpoints/gold",
  "reference":    f"{BASE}/reference"
}

tables = {
  "bronze": f"{UC_CATALOG}.{UC_SCHEMA}.bronze_transactions",
  "silver": f"{UC_CATALOG}.{UC_SCHEMA}.silver_transactions",
  "gold":   f"{UC_CATALOG}.{UC_SCHEMA}.gold_alerts",
  "stats":  f"{UC_CATALOG}.{UC_SCHEMA}.dim_category_stats"
}

# quick sanity check
display(dbutils.fs.ls(BASE))
paths, tables

In [0]:
import pyspark.sql.functions as F

bronze_df = spark.readStream.table(tables["bronze"])

silver_df = (
    bronze_df
    .withColumn(
        "event_time",
        F.coalesce(
            F.to_timestamp("trans_date_trans_time", "M/d/yyyy H:mm"),
            F.to_timestamp("trans_date_trans_time", "M/d/yyyy HH:mm")
        )
    )
    .withColumn("amt", F.col("amt").cast("double"))
    .withColumn("unix_time", F.col("unix_time").cast("long"))
    .withColumn("is_fraud", F.col("is_fraud").cast("int"))
    .withColumn("hour", F.hour("event_time"))
    .withColumn("day_of_week", F.date_format("event_time", "E"))
    .withColumn("state", F.upper(F.col("state")))
    .withColumn("category", F.lower(F.col("category")))
    .withColumn("_silver_ingest_ts", F.current_timestamp())
)

(
    silver_df.writeStream
    .format("delta")
    .option("checkpointLocation", paths["ckpt_silver"])
    .outputMode("append")
    .trigger(availableNow=True)
    .table(tables["silver"])
)

In [0]:
%sql
SELECT
  trans_date_trans_time,
  event_time,
  hour,
  day_of_week
FROM default.silver_transactions
WHERE event_time IS NULL
LIMIT 20;